# From Syntax to Circuits: An Interactive QNLP Tutorial

This notebook builds a small sentiment classifier inspired by Quantum Natural Language Processing (QNLP). The guiding idea is simple: words carry trainable meanings, grammar tells us how those meanings compose, and the composed sentence meaning is read out as a quantum-style measurement probability.

## Setup

The reusable code lives in `src/qnlp_tutorial`. The tutorial is designed to run on CPU. Qiskit is optional for richer circuit rendering; the notebook also includes text and matplotlib visualizations so the core project remains reproducible.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from qnlp_tutorial.data import load_dataset, train_test_split
from qnlp_tutorial.grammar import parse_sentence
from qnlp_tutorial.quantum import ToyQNLPClassifier
from qnlp_tutorial.visualize import (
    save_circuit_diagram,
    save_grammar_diagram,
    save_loss_curve,
    write_poster_outline,
)

## Step 1: The Data

The dataset is intentionally controlled: short positive and negative technical sentences with simple grammar. This keeps the quantum simulation small and makes the relationship between syntax and circuit structure visible.

In [ ]:
dataset = load_dataset()
train, test = train_test_split(dataset)

print(f"Rows: {len(dataset)} | train: {len(train)} | test: {len(test)}")
dataset.head(10)

## Step 2: Grammar to String Diagrams

In DisCoCat-style QNLP, syntax is not just preprocessing. It determines which word meanings interact. Here a deterministic constrained grammar stands in for a full CCG parser so the reductions are easy to inspect.

In [ ]:
example_sentence = "the helpful agent fixes the bug"
diagram = parse_sentence(example_sentence)

print(diagram.ascii_tree())
print()
print(diagram.string_diagram_text())

## Step 3: String Diagrams to Circuits

The toy ansatz uses two qubits: one for the subject phrase and one for the sentence readout. Word parameters become rotation angles, and the grammar decides which rotations contribute to the subject, predicate, and final decision.

In [ ]:
model = ToyQNLPClassifier.from_dataset(dataset)
circuit = model.circuit_for(diagram)

print(circuit.ascii_diagram())
print()
print(circuit.describe())

try:
    qiskit_circuit = circuit.qiskit_circuit()
    print(qiskit_circuit.draw(output="text"))
except ImportError:
    print("Install qiskit for native Qiskit circuit rendering.")

## Step 4: Hybrid Quantum-Classical Training

Training updates word angles to reduce binary cross entropy. The probability of the positive class is interpreted as the measured probability of the sentence qubit. This is a small pedagogical model rather than a hardware-scale quantum advantage claim.

In [ ]:
history = model.fit(train, epochs=50, learning_rate=0.35)
history_df = pd.DataFrame(history)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history_df["epoch"], history_df["loss"], linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Binary cross entropy")
ax.set_title("Training Loss")
ax.grid(True, alpha=0.3)
plt.show()

history_df.tail()

## Step 5: Results and Conclusion

The test set uses held-out sentences with the same constrained grammar. The goal is to show the full pipeline working: syntax-driven composition, circuit construction, trainable parameters, and evaluation.

In [ ]:
train_accuracy = model.accuracy(train)
test_accuracy = model.accuracy(test)

print(f"Train accuracy: {train_accuracy:.1%}")
print(f"Test accuracy: {test_accuracy:.1%}")

model.prediction_table(test)

## Poster Asset Export

The following cell writes the poster visuals referenced in the project plan.

In [ ]:
assets = []
assets.append(save_grammar_diagram(diagram))
assets.append(save_circuit_diagram(model.circuit_for(diagram)))
assets.append(save_loss_curve(history))
assets.append(write_poster_outline(train_accuracy, test_accuracy))

for asset in assets:
    print(asset.relative_to(PROJECT_ROOT))